# 04b — Social Media Charts (Altair + vl-convert)

Publication-ready PNG charts using the @unwelcomedata brand palette.
All charts export to twitter_landscape (1600×900px) with watermark.

**4 Production Charts:**
1. National Abortion Comparison (side-by-side: without vs. with)
2. Top 10 Causes by Sex (stacked bars: male vs. female)
3. Abortion Impact by Race (White)
4. Abortion Impact by Race (Black/African American)

In [ ]:
import sys
import os
from pathlib import Path

import pandas as pd
import duckdb
import yaml
import altair as alt

# Find project root
PROJECT = Path.cwd()
while not (PROJECT / 'config.yaml').exists() and PROJECT != PROJECT.parent:
    PROJECT = PROJECT.parent

os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))
sys.path.insert(0, str(PROJECT.parent / 'shared'))

from src.viz_social import save_social
from viz import PRESETS, SEX_COLORS, PALETTE

with open(PROJECT / 'config.yaml') as f:
    cfg = yaml.safe_load(f)

# Create outputs/social directory if needed
social_dir = PROJECT / 'outputs' / 'social'
social_dir.mkdir(parents=True, exist_ok=True)

# Connect to DuckDB
conn = duckdb.connect(str(PROJECT / 'data' / 'project.duckdb'))

# Get total abortions (national_total measure, 2024)
abort_total = conn.execute('''
  SELECT value
  FROM abortions
  WHERE measure = 'national_total' AND year = 2024
''').df()['value'].iloc[0]


# Display names for cleaner labels
DISPLAY_NAMES = {
    'Diseases of heart': 'Heart disease',
    'Malignant neoplasms': 'Cancer',
    'Chronic lower respiratory diseases': 'Respiratory disease',
    'Cerebrovascular diseases': 'Stroke',
    'Alzheimer disease': "Alzheimer's",
    'Diabetes mellitus': 'Diabetes',
    'Accidents (unintentional injuries)': 'Accidents',
    'Intentional self-harm (suicide)': 'Suicide',
    'Chronic liver disease and cirrhosis': 'Liver disease',
    'Nephritis, nephrotic syndrome and nephrosis': 'Kidney disease',
    'Influenza and pneumonia': 'Flu/Pneumonia',
    'Essential hypertension and hypertensive renal disease': 'Hypertension',
    'Assault (homicide)': 'Homicide',
    'Pregnancy, childbirth and the puerperium': 'Pregnancy/childbirth',
}

def short_name(cause: str) -> str:
    """Map verbose cause name to short display name."""
    return DISPLAY_NAMES.get(cause, cause)

print('✓ Environment loaded')
print(f'✓ Social charts will export to: {social_dir}')
print(f'✓ Total abortions 2024: {abort_total:,.0f}')

## Chart 1: National Abortion Comparison

"If abortion were a cause of death, it would rank as the #2-3 leading cause in the US"

In [ ]:
# Get top 5 causes nationally + abortion total
top_10_national = conn.execute('''
  SELECT 
    cause_raw,
    cause as cause_clean,
    deaths
  FROM mortality_national
  ORDER BY deaths DESC
  LIMIT 5
''').df()

# Apply display names
top_10_national['cause_display'] = top_10_national['cause_clean'].apply(short_name)

print(f'Top 5 causes loaded. #1: {top_10_national.iloc[0]["cause_display"]} ({top_10_national.iloc[0]["deaths"]:,.0f})')
print(f'Abortion total: {abort_total:,.0f} (would rank #{(top_10_national["deaths"] > abort_total).sum() + 1})')

In [ ]:
# Prepare Chart 1 data: stacked male/female bars with abortion inserted
#
# Structure: each cause gets two rows (Male segment, Female segment)
# Abortion gets a single row (solid bar, no sex split)
# We compute x_start and x_end for each segment so bar lengths are accurate.

# Get sex breakdown for all top 10 causes (join on cause_raw)
cause_raw_list = top_10_national['cause_raw'].tolist()
placeholders = ','.join([f"'{c}'" for c in cause_raw_list])
sex_by_cause = conn.execute(f'''
  SELECT 
    icd_10_113_cause_list as cause_raw,
    sex,
    SUM(deaths) as deaths
  FROM mortality_sex_age
  WHERE icd_10_113_cause_list IN ({placeholders})
  GROUP BY icd_10_113_cause_list, sex
''').df()

# Build the chart data: one entry per segment (Male/Female/Abortion)
rows = []
for _, row in top_10_national.iterrows():
    cause_raw = row['cause_raw']
    cause_display = row['cause_display']
    total_deaths = int(row['deaths'])
    
    sex_data = sex_by_cause[sex_by_cause['cause_raw'] == cause_raw]
    male_d = int(sex_data[sex_data['sex'] == 'Male']['deaths'].sum())
    female_d = int(sex_data[sex_data['sex'] == 'Female']['deaths'].sum())
    sex_total = male_d + female_d
    
    # Use sex_total for bar length (segments add up correctly)
    male_pct = round(100 * male_d / sex_total) if sex_total > 0 else 0
    female_pct = round(100 * female_d / sex_total) if sex_total > 0 else 0
    
    # Male segment: starts at 0, ends at male_d
    rows.append({
        'cause': cause_display,
        'segment': 'Male',
        'x_start': 0,
        'x_end': male_d,
        'seg_deaths': male_d,
        'total_deaths': total_deaths,
        'pct_label': f'{male_pct}%',
        'is_abortion': False,
    })
    # Female segment: starts at male_d, ends at sex_total
    rows.append({
        'cause': cause_display,
        'segment': 'Female',
        'x_start': male_d,
        'x_end': sex_total,
        'seg_deaths': female_d,
        'total_deaths': total_deaths,
        'pct_label': f'{female_pct}%',
        'is_abortion': False,
    })

# Insert Abortion row (single segment, no sex split)
rows.append({
    'cause': 'Abortion',
    'segment': 'Abortion',
    'x_start': 0,
    'x_end': int(abort_total),
    'seg_deaths': int(abort_total),
    'total_deaths': int(abort_total),
    'pct_label': '',  # No sex label for abortion
    'is_abortion': True,
})

df_chart1 = pd.DataFrame(rows)

# Compute midpoint for centering text inside each segment
df_chart1['x_mid'] = (df_chart1['x_start'] + df_chart1['x_end']) / 2

# Sort order: largest total at top (descending)
cause_totals = df_chart1.groupby('cause')['total_deaths'].first().reset_index()
cause_totals = cause_totals.sort_values('total_deaths', ascending=False)
cause_order = cause_totals['cause'].tolist()

# Build a separate df for total labels (one row per cause, at x_end of full bar)
df_totals = df_chart1.groupby('cause').agg(
    total_deaths=('total_deaths', 'first'),
    bar_end=('x_end', 'max')
).reset_index()
df_totals['total_label'] = df_totals['total_deaths'].apply(lambda x: f'{x:,}')

print(f'Chart 1 data prepared:')
print(f'  {len(cause_order)} causes (including Abortion)')
print(f'  Abortion: {abort_total:,.0f}')
print(f'  Heart disease: {top_10_national.iloc[0]["deaths"]:,.0f}')
print(f'\nSegment data sample:')
print(df_chart1[['cause', 'segment', 'x_start', 'x_end', 'pct_label']].head(6))
print(f'\nCause order (bottom to top): {cause_order}')

In [ ]:
# Build Chart 1: Stacked male/female bars with sex% inside, total count to right
# Uses x (start) and x2 (end) encoding for accurate proportional bar segments.

from viz import SEX_COLORS

# Color mapping: Male=light teal, Female=peach, Abortion=burnt caramel
segment_colors = {
    'Male': '#005F73',
    'Female': '#E9D8A6',
    'Abortion': '#AE2012',
}

# --- Layer 1: Stacked bars using x/x2 ---
bars = alt.Chart(df_chart1).mark_bar().encode(
    y=alt.Y('cause:N', title='', sort=cause_order,
            axis=alt.Axis(labelFontSize=14, labelFontWeight='bold', domain=False, ticks=False)),
    x=alt.X('x_start:Q', title='', axis=None, scale=alt.Scale(domain=[0, df_chart1['x_end'].max() * 1.12])),
    x2='x_end:Q',
    color=alt.Color('segment:N',
                    scale=alt.Scale(
                        domain=['Male', 'Female', 'Abortion'],
                        range=[segment_colors['Male'], segment_colors['Female'], segment_colors['Abortion']]
                    ),
                    legend=None),
).properties(
    width=1300,
    height=500,
    title={
        'text': 'What if abortion was counted as a cause of death?',
        'subtitle': 'Top 5 causes of death among all Americans, by sex (2024)',
        'anchor': 'start',
        'offset': 10,
    }
)

# --- Layer 2: Direct 'Male' / 'Female' labels inside Heart disease bar ---
heart_row = df_chart1[df_chart1['cause'] == 'Heart disease']
heart_male = heart_row[heart_row['segment'] == 'Male'].iloc[0]
heart_female = heart_row[heart_row['segment'] == 'Female'].iloc[0]

# Male label (light text on dark teal)
df_label_male = pd.DataFrame([{'cause': 'Heart disease', 'x_pos': heart_male['x_start'], 'label': 'Male'}])
text_label_male = alt.Chart(df_label_male).mark_text(
    align='left', baseline='top', dx=8, dy=-20,
    fontSize=13, fontWeight='bold', color='#E9D8A6'
).encode(y=alt.Y('cause:N', sort=cause_order), x=alt.X('x_pos:Q'), text='label:N')

# Female label (dark text on vanilla custard)
df_label_female = pd.DataFrame([{'cause': 'Heart disease', 'x_pos': heart_female['x_start'], 'label': 'Female'}])
text_label_female = alt.Chart(df_label_female).mark_text(
    align='left', baseline='top', dx=8, dy=-20,
    fontSize=13, fontWeight='bold', color='#003049'
).encode(y=alt.Y('cause:N', sort=cause_order), x=alt.X('x_pos:Q'), text='label:N')

# --- Layer 3: Sex percentage labels centered inside each segment ---
# Split into two layers for proper text color contrast
max_bar = df_chart1['x_end'].max()
min_segment_for_label = max_bar * 0.035
df_pct_labels = df_chart1[
    (df_chart1['is_abortion'] == False) &
    (df_chart1['seg_deaths'] >= min_segment_for_label)
].copy()

# Male pct labels (light text on dark teal)
df_pct_male = df_pct_labels[df_pct_labels['segment'] == 'Male']
text_pct_male = alt.Chart(df_pct_male).mark_text(
    align='center', baseline='middle', color='#E9D8A6', fontSize=13, fontWeight='bold'
).encode(y=alt.Y('cause:N', sort=cause_order), x=alt.X('x_mid:Q'), text='pct_label:N')

# Female pct labels (dark text on vanilla custard)
df_pct_female = df_pct_labels[df_pct_labels['segment'] == 'Female']
text_pct_female = alt.Chart(df_pct_female).mark_text(
    align='center', baseline='middle', color='#003049', fontSize=13, fontWeight='bold'
).encode(y=alt.Y('cause:N', sort=cause_order), x=alt.X('x_mid:Q'), text='pct_label:N')

# --- Layer 3: Total count label to the right of each bar ---
text_total = alt.Chart(df_totals).mark_text(
    align='left', baseline='middle', dx=8,
    fontSize=15, fontWeight='bold', color='#374151'
).encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('bar_end:Q'),
    text='total_label:N',
)

# --- Combine all layers ---
# First, add gestation age dividers within the abortion bar
# Get gestation data: <=9wk (78.6%), 10-13wk (14.2%), 14-20wk (6.1%), >=21wk (1.1%)
gest_data = conn.execute("""
    SELECT age_group, value FROM abortions
    WHERE measure = 'gestation_count' ORDER BY value DESC
""").df()
gest_pcts = conn.execute("""
    SELECT age_group, value FROM abortions
    WHERE measure = 'gestation_pct' ORDER BY value DESC
""").df()

# Build gestation segments (cumulative x positions, ordered earliest first)
gest_order = ['<=9_weeks', '10-13_weeks', '14-20_weeks', '>=21_weeks']
gest_labels = {'<=9_weeks': '≤9 wks', '10-13_weeks': '10-13 wks', '14-20_weeks': '14-20 wks', '>=21_weeks': '≥21 wks'}
x_cursor = 0
gest_boundaries = []  # x positions for vertical rules
gest_label_rows = []  # for centered text labels
for g in gest_order:
    count = int(gest_data[gest_data['age_group'] == g]['value'].iloc[0])
    pct = float(gest_pcts[gest_pcts['age_group'] == g]['value'].iloc[0])
    x_start_g = x_cursor
    x_end_g = x_cursor + count
    x_mid_g = (x_start_g + x_end_g) / 2
    gest_label_rows.append({'cause': 'Abortion', 'x_mid': x_mid_g, 'label': gest_labels[g], 'pct': f'{pct:.0f}%'}) if count > mx * 0.10 else None
    x_cursor = x_end_g
    if x_cursor < int(abort_total):  # don't add rule after last segment
        gest_boundaries.append({'cause': 'Abortion', 'x_pos': x_cursor})

# Vertical rules at gestation boundaries (thick dark red lines)
df_gest_rules = pd.DataFrame(gest_boundaries)
# Add a tiny x_end for each rule (3px worth of the data scale)
px_width = (df_chart1['x_end'].max() * 1.12) / 1300 * 3  # ~3px in data units
df_gest_rules['x_end'] = df_gest_rules['x_pos'] + px_width
gest_rules = alt.Chart(df_gest_rules).mark_bar(
    color='#fff5e6',
).encode(
    x=alt.X('x_pos:Q'),
    x2='x_end:Q',
    y=alt.Y('cause:N', sort=cause_order),
) if len(df_gest_rules) > 0 else alt.Chart(pd.DataFrame()).mark_point()

# Gestation labels centered in each section
df_gest_labels = pd.DataFrame(gest_label_rows)
# Gestation name (above center)
gest_text_name = alt.Chart(df_gest_labels).mark_text(
    align='center', baseline='bottom', dy=-2, color='#E9D8A6', fontSize=11, fontWeight='bold'
).encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('x_mid:Q'),
    text='label:N',
)
# Gestation pct (below center)
gest_text_pct = alt.Chart(df_gest_labels).mark_text(
    align='center', baseline='top', dy=2, color='#E9D8A6', fontSize=11, fontWeight='bold'
).encode(
    y=alt.Y('cause:N', sort=cause_order),
    x=alt.X('x_mid:Q'),
    text='pct:N',
)

chart1 = (bars + text_label_male + text_label_female + text_pct_male + text_pct_female + text_total + gest_rules + gest_text_name + gest_text_pct).configure_axis(
    grid=False,
    domain=False,
    labelColor='#374151'
).configure_view(
    strokeWidth=0
)

print('Chart 1 built: stacked male/female bars + sex% inside + total to right')

In [ ]:
# Export Chart 1
from chart_templates import add_footer
chart1_final = add_footer(chart1, source='CDC WONDER 2024 (mortality) | Guttmacher Institute 2024 (abortion counts)')
save_social(chart1_final, cfg, '01_abortion_comparison_national', preset='twitter_landscape')
print('✓ Chart 1 exported')

from IPython.display import Image, display
display(Image(filename=str(social_dir / '01_abortion_comparison_national.png')))

## Charts 2 & 3: Top 5 Causes by Race (White & Black)

Same stacked male/female bar pattern, filtered by race.
Uses the  template with footer.

In [ ]:
# Charts 2 & 3: Top 5 causes by race (White, Black) — same format as Chart 1 with abortion
from chart_templates import add_footer

races = [
    ("White", "02_top5_causes_white"),
    ("Black or African American", "03_top5_causes_black"),
]
RACE_DISPLAY = {"White": "White", "Black or African American": "Black"}

# Race-specific abortion counts from DuckDB
# Source: Guttmacher Abortion Patient Survey 2021-2022 proportions applied to 2024 total
ABORT_BY_RACE = {
    "White": int(conn.execute("SELECT value FROM abortions WHERE measure='race_count' AND age_group='NH White'").df()['value'].iloc[0]),
    "Black or African American": int(conn.execute("SELECT value FROM abortions WHERE measure='race_count' AND age_group='Black'").df()['value'].iloc[0]),
}

for race_name, filename in races:
    race_display = RACE_DISPLAY[race_name]
    # Get top 5 causes for this race (only # prefixed = summary causes)
    top5_race = conn.execute(f"""
        SELECT icd_10_113_cause_list as cause_raw, SUM(deaths) as total_deaths
        FROM mortality_race_sex
        WHERE single_race_6 = '{race_name}'
          AND icd_10_113_cause_list LIKE '#%'
        GROUP BY icd_10_113_cause_list
        ORDER BY total_deaths DESC
        LIMIT 5
    """).df()

    # Get sex breakdown for those causes
    cause_list = top5_race["cause_raw"].tolist()
    ph = ",".join([f"'{c}'" for c in cause_list])
    sex_race = conn.execute(f"""
        SELECT icd_10_113_cause_list as cause_raw, sex, SUM(deaths) as deaths
        FROM mortality_race_sex
        WHERE single_race_6 = '{race_name}'
          AND icd_10_113_cause_list IN ({ph})
        GROUP BY icd_10_113_cause_list, sex
    """).df()

    # Build segment data (same structure as Chart 1)
    rows = []
    for _, row in top5_race.iterrows():
        cause_clean = row["cause_raw"].lstrip("#").split(" (")[0]
        cause_display = short_name(cause_clean)
        total_deaths = int(row["total_deaths"])
        sd = sex_race[sex_race["cause_raw"] == row["cause_raw"]]
        male_d = int(sd[sd["sex"] == "Male"]["deaths"].sum())
        female_d = int(sd[sd["sex"] == "Female"]["deaths"].sum())
        sex_total = male_d + female_d
        male_pct = round(100 * male_d / sex_total) if sex_total > 0 else 0
        female_pct = round(100 * female_d / sex_total) if sex_total > 0 else 0
        rows.append({"cause": cause_display, "segment": "Male", "x_start": 0, "x_end": male_d,
                     "seg_deaths": male_d, "total_deaths": total_deaths, "pct_label": f"{male_pct}%", "is_abortion": False})
        rows.append({"cause": cause_display, "segment": "Female", "x_start": male_d, "x_end": sex_total,
                     "seg_deaths": female_d, "total_deaths": total_deaths, "pct_label": f"{female_pct}%", "is_abortion": False})

    # Insert Abortion row
    rows.append({"cause": "Abortion", "segment": "Abortion", "x_start": 0, "x_end": ABORT_BY_RACE[race_name],
                 "seg_deaths": ABORT_BY_RACE[race_name], "total_deaths": ABORT_BY_RACE[race_name], "pct_label": "", "is_abortion": True})

    df_race = pd.DataFrame(rows)
    df_race["x_mid"] = (df_race["x_start"] + df_race["x_end"]) / 2

    # Sort: largest on top
    ct = df_race.groupby("cause")["total_deaths"].first().reset_index().sort_values("total_deaths", ascending=False)
    co = ct["cause"].tolist()
    df_t = df_race.groupby("cause").agg(total_deaths=("total_deaths", "first"), bar_end=("x_end", "max")).reset_index()
    df_t["total_label"] = df_t["total_deaths"].apply(lambda x: f"{x:,}")

    segment_colors = {"Male": "#005F73", "Female": "#E9D8A6", "Abortion": "#AE2012"}

    bars_r = alt.Chart(df_race).mark_bar().encode(
        y=alt.Y("cause:N", title="", sort=co, axis=alt.Axis(labelFontSize=14, labelFontWeight="bold", domain=False, ticks=False)),
        x=alt.X("x_start:Q", title="", axis=None, scale=alt.Scale(domain=[0, df_race["x_end"].max() * 1.12])),
        x2="x_end:Q",
        color=alt.Color("segment:N", scale=alt.Scale(domain=list(segment_colors.keys()), range=list(segment_colors.values())), legend=None),
    ).properties(width=1300, height=500,
        title={"text": "What if abortion was counted as a cause of death?",
               "subtitle": f"Top 5 causes of death among {race_display} Americans, by sex (2024)",
               "anchor": "start", "offset": 10})

    # Direct labels on first non-abortion cause
    first_cause = co[1] if co[0] == "Abortion" else co[0]
    fc_rows = df_race[df_race["cause"] == first_cause]
    fc_m = fc_rows[fc_rows["segment"] == "Male"].iloc[0]
    fc_f = fc_rows[fc_rows["segment"] == "Female"].iloc[0]

    tl_m = alt.Chart(pd.DataFrame([{"cause": first_cause, "x_pos": fc_m["x_start"], "label": "Male"}])).mark_text(
        align="left", baseline="top", dx=8, dy=-20, fontSize=13, fontWeight="bold", color="#E9D8A6"
    ).encode(y=alt.Y("cause:N", sort=co), x="x_pos:Q", text="label:N")

    tl_f = alt.Chart(pd.DataFrame([{"cause": first_cause, "x_pos": fc_f["x_start"], "label": "Female"}])).mark_text(
        align="left", baseline="top", dx=8, dy=-20, fontSize=13, fontWeight="bold", color="#003049"
    ).encode(y=alt.Y("cause:N", sort=co), x="x_pos:Q", text="label:N")

    # Pct labels (split for text color contrast)
    mx = df_race["x_end"].max()
    mn = mx * 0.035
    dpl = df_race[(~df_race["is_abortion"]) & (df_race["seg_deaths"] >= mn)]
    tp_m = alt.Chart(dpl[dpl["segment"] == "Male"]).mark_text(
        align="center", baseline="middle", color="#E9D8A6", fontSize=13, fontWeight="bold"
    ).encode(y=alt.Y("cause:N", sort=co), x="x_mid:Q", text="pct_label:N")
    tp_f = alt.Chart(dpl[dpl["segment"] == "Female"]).mark_text(
        align="center", baseline="middle", color="#003049", fontSize=13, fontWeight="bold"
    ).encode(y=alt.Y("cause:N", sort=co), x="x_mid:Q", text="pct_label:N")
    tt = alt.Chart(df_t).mark_text(
        align="left", baseline="middle", dx=8, fontSize=15, fontWeight="bold", color="#374151"
    ).encode(y=alt.Y("cause:N", sort=co), x="bar_end:Q", text="total_label:N")

    chart_r = (bars_r + tl_m + tl_f + tp_m + tp_f + tt).configure_axis(
        grid=False, domain=False, labelColor="#374151"
    ).configure_view(strokeWidth=0)

    chart_r_final = add_footer(chart_r, source="CDC WONDER 2024 (mortality) | Guttmacher Institute 2024 (abortion counts)")
    save_social(chart_r_final, cfg, filename, preset="twitter_landscape")
    print(f"✓ {filename} exported")

    from IPython.display import Image, display
    display(Image(filename=str(social_dir / f"{filename}.png")))


## Chart 2: Top 10 Causes by Sex

In [ ]:
# Get top 10 causes from national table
top_causes = conn.execute('''
  SELECT cause
  FROM mortality_national
  ORDER BY deaths DESC
  LIMIT 10
''').df()['cause'].tolist()

# Get sex breakdown for each
sex_breakdown = conn.execute(f'''
  SELECT 
    COALESCE(SUBSTR(icd_10_113_cause_list, 2), icd_10_113_cause_list) as cause,
    sex,
    SUM(deaths) as deaths
  FROM mortality_sex_age
  WHERE icd_10_113_cause_list IN ({','.join([f"'{c}'" for c in top_causes])})
  GROUP BY icd_10_113_cause_list, sex
  ORDER BY icd_10_113_cause_list, sex
''').df()

# Pivot and sort
sex_pivot = sex_breakdown.pivot_table(
    index='cause', columns='sex', values='deaths', aggfunc='sum'
).reset_index()

# Ensure both columns exist
sex_pivot['Female'] = sex_pivot.get('Female', 0)
sex_pivot['Male'] = sex_pivot.get('Male', 0)

sex_pivot['total'] = sex_pivot['Female'] + sex_pivot['Male']
sex_pivot = sex_pivot.sort_values('total', ascending=True).reset_index(drop=True)

# Melt for stacked bar
sex_long = sex_pivot[['cause', 'Female', 'Male']].melt(
    id_vars=['cause'],
    value_vars=['Female', 'Male'],
    var_name='sex',
    value_name='deaths'
)

print(f'Sex breakdown ready: {len(sex_long)} rows')

In [ ]:
# Build Chart 2: Stacked bars
color_map = {'Male': SEX_COLORS.get('Male', '#005F73'), 'Female': SEX_COLORS.get('Female', '#AE2012')}

chart2 = alt.Chart(sex_long).mark_bar().encode(
    y=alt.Y('cause:N', title='', sort=sex_pivot['cause'].tolist()),
    x=alt.X('deaths:Q', title='Deaths (2024)', stack='zero'),
    color=alt.Color('sex:N', scale=alt.Scale(
        domain=['Female', 'Male'],
        range=[color_map['Female'], color_map['Male']]
    ), title='Sex'),
    tooltip=['cause', 'sex', 'deaths'],
).properties(
    width=1450,
    height=720,
    title={
        'text': 'Leading Causes of Death by Sex (2024)',
        'subtitle': 'Top 10 causes nationally, broken down by male and female deaths',
        'anchor': 'start',
        'offset': 10,
    }
).configure_axis(
    labelFontSize=10,
    titleFontSize=11,
    labelColor='#374151',
    titleColor='#374151'
).configure_legend(
    labelFontSize=11,
    orient='top',
)

print('Chart 2 created')
chart2

In [ ]:
# Export Chart 2
save_social(chart2, cfg, '02_top_10_causes_by_sex', preset='twitter_landscape')
print('✓ Chart 2 exported')

## Charts 3 & 4: Abortion by Race

In [ ]:
races_to_chart = ['White', 'Black or African American']
charts_by_race = {}

for idx, race in enumerate(races_to_chart, 1):
    print(f'Building Chart {2+idx}: {race}...')
    
    # Get top 10 causes for this race
    top_10_race = conn.execute(f'''
      SELECT 
        COALESCE(SUBSTR(icd_10_113_cause_list, 2), icd_10_113_cause_list) as cause,
        SUM(deaths) as deaths
      FROM mortality_race_sex
      WHERE single_race_6 = '{race}'
      GROUP BY icd_10_113_cause_list
      ORDER BY deaths DESC
      LIMIT 10
    ''').df()
    
    df_without_race = top_10_race.copy()
    df_without_race['comparison'] = 'Without Abortion'
    
    # Create with abortion
    df_with_race_list = []
    rank = 1
    for row_idx, row_tuple in enumerate(top_10_race.itertuples(), 1):
        if rank <= 2 and abort_total > row_tuple.deaths:
            df_with_race_list.append({'cause': 'Induced Abortion', 'deaths': abort_total, 'comparison': 'With Abortion'})
            rank += 1
        df_with_race_list.append({'cause': row_tuple.cause, 'deaths': row_tuple.deaths, 'comparison': 'With Abortion'})
        rank += 1
        if len(df_with_race_list) >= 11:
            break
    
    df_with_race = pd.DataFrame(df_with_race_list)
    df_race_combined = pd.concat([df_without_race, df_with_race], ignore_index=True)
    
    cause_order_race = df_without_race.sort_values('deaths', ascending=True)['cause'].tolist()
    
    # Build chart
    chart = alt.Chart(df_race_combined).mark_bar().encode(
        y=alt.Y('cause:N', title='', sort=cause_order_race),
        x=alt.X('deaths:Q', title='Deaths (2024)'),
        color=alt.Color('comparison:N', scale=alt.Scale(
            domain=['Without Abortion', 'With Abortion'],
            range=[PALETTE['cat_2'], PALETTE['accent']]
        ), title=''),
        xOffset='comparison:N',
        tooltip=['cause', 'comparison', 'deaths'],
    ).properties(
        width=1450,
        height=720,
        title={
            'text': f'If Abortion Were a Leading Cause: {race}',
            'subtitle': 'How abortion would rank among top 10 causes of death (2024)',
            'anchor': 'start',
            'offset': 10,
        }
    ).configure_axis(
        labelFontSize=10,
        titleFontSize=11,
        labelColor='#374151',
        titleColor='#374151'
    ).configure_legend(
        labelFontSize=11,
        orient='top',
    )
    
    charts_by_race[race] = chart
    print(f'Chart {2+idx} created')

In [ ]:
# Export race charts
safe_names = {
    'White': 'white',
    'Black or African American': 'black_or_african_american'
}

for idx, race in enumerate(races_to_chart, 1):
    chart = charts_by_race[race]
    safe_name = safe_names[race]
    save_social(chart, cfg, f'0{2+idx}_abortion_comparison_race_{safe_name}', preset='twitter_landscape')
    print(f'✓ Chart {2+idx} ({race}) exported')

## Summary & Cleanup

In [ ]:
conn.close()

# Verify all exports
pngs = sorted(social_dir.glob('*.png'))
print('=== ALL CHARTS COMPLETE ===')
print(f'\n✓ Generated {len(pngs)} publication-ready charts:')
for png in pngs:
    size_kb = png.stat().st_size / 1024
    print(f'  • {png.name} ({size_kb:.0f} KB)')

print('\nAll charts are twitter_landscape (1600×900px) with @unwelcomedata watermark.')
print('Ready for social media posting!')